In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q -r /content/drive/MyDrive/ai-study-companion/backend/requirements.txt

In [3]:
import pydantic
import pydantic_settings
import fastapi
import pymongo
import groq
import sentence_transformers

print("✓ Backend dependencies installed")

✓ Backend dependencies installed


In [4]:
# ============================================================
# AI STUDY COMPANION
# SINGLE BACKEND LAUNCHER
# ============================================================

from pathlib import Path
import sys
import types
import json
import threading
import time


# ============================================================
# PROJECT PATHS
# ============================================================

import os

PROJECT_ROOT = Path(
    os.getenv(
        "PROJECT_ROOT",
        "/content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/",
    )
)

BACKEND_DIR = PROJECT_ROOT / "backend"

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))


print("=" * 70)
print("AI STUDY COMPANION — BACKEND LAUNCHER")
print("=" * 70)
print(f"Project: {PROJECT_ROOT}")


# ============================================================
# CLEAN PREVIOUS LOADED APPLICATION MODULES
# ============================================================

# Important for Colab.
# Prevents stale app.* modules from previous launcher runs.

for module_name in list(sys.modules.keys()):
    if (
        module_name == "app"
        or module_name.startswith("app.")
    ):
        del sys.modules[module_name]


# ============================================================
# CREATE PACKAGE HIERARCHY
# ============================================================

def ensure_package_hierarchy():

    packages = [
        "app",
        "app.ai",
        "app.api",
        "app.core",
        "app.models",
        "app.schemas",
        "app.services",
        "app.utils",
        "app.workers",
    ]

    for package_name in packages:

        if package_name not in sys.modules:

            package = types.ModuleType(
                package_name
            )

            package.__path__ = []
            package.__package__ = package_name

            sys.modules[package_name] = package


ensure_package_hierarchy()


# ============================================================
# NOTEBOOK SOURCE LOADER
# ============================================================
def normalize_cell_source(source):
    if isinstance(source, str):
        return source

    if isinstance(source, list):
        parts = []

        def collect(value):
            if isinstance(value, str):
                parts.append(value)
            elif isinstance(value, list):
                for item in value:
                    collect(item)

        collect(source)
        return "".join(parts)

    return ""

def notebook_source(notebook_path):
    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    sources = []

    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue

        source = normalize_cell_source(cell.get("source", []))

        if not source.strip():
            continue

        cleaned_lines = []

        for line in source.splitlines(keepends=True):
            stripped = line.lstrip()

            # Ignore Jupyter/Colab magic commands
            if stripped.startswith("%"):
                continue

            # Ignore shell commands
            if stripped.startswith("!"):
                continue

            cleaned_lines.append(line)

        cleaned = "".join(cleaned_lines)

        if cleaned.strip():
            sources.append(cleaned)

    return "\n\n".join(sources)


def load_project_module(
    relative_path: str,
    module_name: str,
):

    notebook_path = (
        PROJECT_ROOT / relative_path
    )

    if not notebook_path.exists():

        raise FileNotFoundError(
            f"Notebook not found: {notebook_path}"
        )

    # Remove stale module if present.
    sys.modules.pop(
        module_name,
        None
    )

    module = types.ModuleType(
        module_name
    )

    module.__file__ = str(
        notebook_path
    )

    module.__package__ = (
        module_name.rpartition(".")[0]
    )

    sys.modules[module_name] = module

    source = notebook_source(
        notebook_path
    )

    compiled = compile(
        source,
        str(notebook_path),
        "exec",
    )

    exec(
        compiled,
        module.__dict__,
    )

    return module


# ============================================================
# LOAD BACKEND MODULES
# ============================================================

print()
print("=" * 70)
print("LOADING BACKEND MODULES")
print("=" * 70)


# ------------------------------------------------------------
# CORE
# ------------------------------------------------------------

core_modules = [
    (
        "backend/app/core/config.ipynb",
        "app.core.config",
    ),
    (
        "backend/app/core/security.ipynb",
        "app.core.security",
    ),
    (
        "backend/app/core/database.ipynb",
        "app.core.database",
    ),
]


# ------------------------------------------------------------
# UTILS
# ------------------------------------------------------------

utils_modules = [
    (
        "backend/app/utils/helpers.ipynb",
        "app.utils.helpers",
    ),
    (
        "backend/app/utils/logging.ipynb",
        "app.utils.logging",
    ),
]


# ------------------------------------------------------------
# MODELS
# ------------------------------------------------------------

model_names = [
    "activity",
    "ai_usage",
    "assessment",
    "concept",
    "conversation",
    "document_chunk",
    "mastery",
    "material",
    "project",
    "recommendation",
    "space",
    "user",
]

model_modules = [
    (
        f"backend/app/models/{name}.ipynb",
        f"app.models.{name}",
    )
    for name in model_names
]


# ------------------------------------------------------------
# SCHEMAS
# ------------------------------------------------------------

schema_names = [
    "analytics",
    "auth",
    "material",
    "project",
    "quiz",
    "space",
    "tutor",
]

schema_modules = [
    (
        f"backend/app/schemas/{name}.ipynb",
        f"app.schemas.{name}",
    )
    for name in schema_names
]


# ------------------------------------------------------------
# AI
# ------------------------------------------------------------

ai_names = [
    "prompts",
    "tutor",
    "quiz_generator",
    "evaluator",
    "recommender",
]

ai_modules = [
    (
        f"backend/app/ai/{name}.ipynb",
        f"app.ai.{name}",
    )
    for name in ai_names
]


# ------------------------------------------------------------
# SERVICES
# ------------------------------------------------------------

service_names = [
    "ai_service",
    "document_service",
    "retrieval_service",
    "tutor_service",
    "quiz_service",
    "assessment_service",
    "mastery_service",
    "recommendation_service",
    "analytics_service",
    "growth_service",
]

service_modules = [
    (
        f"backend/app/services/{name}.ipynb",
        f"app.services.{name}",
    )
    for name in service_names
]


# ------------------------------------------------------------
# WORKERS
# ------------------------------------------------------------

worker_names = [
    "document_worker",
    "learning_worker",
    "quiz_worker",
]

worker_modules = [
    (
        f"backend/app/workers/{name}.ipynb",
        f"app.workers.{name}",
    )
    for name in worker_names
]


module_groups = [
    ("core", core_modules),
    ("utils", utils_modules),
    ("models", model_modules),
    ("schemas", schema_modules),
    ("ai", ai_modules),
    ("services", service_modules),
    ("workers", worker_modules),
]


loaded_modules = {}

for group_name, modules in module_groups:

    print()
    print(f"Loading {group_name}...")

    for relative_path, module_name in modules:

        loaded_modules[module_name] = (
            load_project_module(
                relative_path,
                module_name,
            )
        )

        # Runtime fingerprint for debugging notebook/source mismatches.
        if module_name in {
            "app.ai.quiz_generator",
            "app.services.quiz_service",
            "app.services.ai_service",
        }:
            loaded = loaded_modules[module_name]
            versions = [
                value
                for name, value in vars(loaded).items()
                if name.endswith("_VERSION")
            ]
            print(
                f"  [runtime] {module_name}: "
                f"{loaded.__file__} | version={versions[0] if versions else 'unknown'}"
            )

    print(
        f"✓ {group_name} loaded"
    )


# ============================================================
# LOAD THE SINGLE FASTAPI APPLICATION
# ============================================================

print()
print("=" * 70)
print("LOADING FASTAPI APPLICATION")
print("=" * 70)

main_module = load_project_module(
    "backend/app/main.ipynb",
    "app.main",
)

app = main_module.app
import traceback
from fastapi.responses import JSONResponse

@app.exception_handler(Exception)
async def debug_exception_handler(request, exc):
    print("\n" + "=" * 70)
    print("UNHANDLED BACKEND EXCEPTION")
    print("=" * 70)
    print(f"Request: {request.method} {request.url}")
    print(f"Exception: {type(exc).__name__}: {exc}")
    traceback.print_exc()
    print("=" * 70 + "\n")

    return JSONResponse(
        status_code=500,
        content={
            "detail": "Internal server error",
            "error_type": type(exc).__name__,
            "error": str(exc),
        },
    )

print(
    f"FastAPI app object: {id(app)}"
)

# The app that Uvicorn serves must be the one carrying CORS.
from fastapi.middleware.cors import CORSMiddleware

cors_middleware = next(
    (
        middleware
        for middleware in app.user_middleware
        if middleware.cls is CORSMiddleware
    ),
    None,
)

if cors_middleware is None:
    raise RuntimeError(
        "CORSMiddleware is not attached to the served FastAPI app."
    )

print(
    "CORS origins:",
    cors_middleware.kwargs.get("allow_origins"),
)


# ============================================================
# INITIALIZE MONGODB
# ============================================================

print()
print("=" * 70)
print("INITIALIZING MONGODB")
print("=" * 70)

from app.core.config import settings
from app.core.database import create_database


database = create_database(
    mongodb_uri=settings.mongodb_uri,
    database_name=settings.mongodb_database,
)

app.state.database = database

print("✓ MongoDB initialized")


# ============================================================
# LOAD API ROUTERS
# ============================================================

api_modules = [
    (
        "auth",
        "backend/app/api/auth.ipynb",
    ),
    (
        "spaces",
        "backend/app/api/spaces.ipynb",
    ),
    (
        "projects",
        "backend/app/api/projects.ipynb",
    ),
    (
        "materials",
        "backend/app/api/materials.ipynb",
    ),
    (
        "tutor",
        "backend/app/api/tutor.ipynb",
    ),
    (
        "quiz",
        "backend/app/api/quiz.ipynb",
    ),
    (
        "mastery",
        "backend/app/api/mastery.ipynb",
    ),
    (
        "growth",
        "backend/app/api/growth.ipynb",
    ),
    (
        "analytics",
        "backend/app/api/analytics.ipynb",
    ),
    (
        "admin",
        "backend/app/api/admin.ipynb",
    ),
    (
        "recommendations",
        "backend/app/api/recommendations.ipynb",
    ),
]


# ============================================================
# REGISTER API ROUTERS
# ============================================================

print()
print("=" * 70)
print("REGISTERING API ROUTERS")
print("=" * 70)

loaded_api_modules = {}

for name, relative_path in api_modules:

    module_name = (
        f"app.api.{name}"
    )

    module = load_project_module(
        relative_path,
        module_name,
    )

    loaded_api_modules[name] = module

    router = getattr(
        module,
        "router",
        None,
    )

    if router is None:

        raise RuntimeError(
            f"API module '{name}' "
            f"does not expose a router."
        )

    route_count = len(
        router.routes
    )

    print(
        f"✓ {name:12} "
        f"prefix={router.prefix:12} "
        f"routes={route_count}"
    )

    if route_count == 0:

        raise RuntimeError(
            f"API router '{name}' "
            f"is empty."
        )

    # All API routes receive the
    # central /api prefix.
    app.include_router(
        router,
        prefix="/api",
    )


# ============================================================
# VERIFY FINAL FASTAPI ROUTES
# ============================================================

print()
print("=" * 70)
print("FINAL FASTAPI ROUTES")
print("=" * 70)

route_paths = []

for route in app.routes:

    path = getattr(
        route,
        "path",
        None,
    )

    methods = getattr(
        route,
        "methods",
        set(),
    )

    if path:

        route_paths.append(
            (
                path,
                sorted(methods),
            )
        )

        print(
            f"{','.join(sorted(methods)):20} "
            f"{path}"
        )

print("=" * 70)
print("ROUTE VALIDATION")
print("=" * 70)

# ------------------------------------------------------------------
# IMPORTANT:
# FastAPI 0.141.1 stores included routers in app.routes as
# _IncludedRouter objects. Therefore app.routes does NOT directly
# expose every endpoint.
#
# Validate the original APIRouter objects instead.
# ------------------------------------------------------------------

expected_api_routes = {
    # Authentication
    "GET /api/auth/me",
    "POST /api/auth/register",
    "POST /api/auth/login",

    # Spaces
    "GET /api/spaces",
    "POST /api/spaces",
    "GET /api/spaces/{space_id}",
    "PUT /api/spaces/{space_id}",
    "DELETE /api/spaces/{space_id}",

    # Projects
    "GET /api/projects",
    "POST /api/projects",
    "GET /api/projects/{project_id}",
    "PUT /api/projects/{project_id}",
    "DELETE /api/projects/{project_id}",

    # Materials
    "POST /api/materials/upload",
    "GET /api/materials/project/{project_id}",
    "GET /api/materials/{material_id}",
    "DELETE /api/materials/{material_id}",

    # Tutor
    "POST /api/tutor/ask",
    "GET /api/tutor/conversations/project/{project_id}",
    "GET /api/tutor/conversations/{conversation_id}",

    # Quiz
    "POST /api/quiz/generate",
    "POST /api/quiz/{assessment_id}/start",
    "POST /api/quiz/{assessment_id}/answer",
    "GET /api/quiz/{assessment_id}",
    "POST /api/quiz/{assessment_id}/complete",

    # Mastery
    "GET /api/mastery/project/{project_id}",
    "GET /api/mastery/project/{project_id}/concept/{concept_id}",

    # Growth
    "GET /api/growth/project/{project_id}",

    # Analytics
    "GET /api/analytics/project/{project_id}",
    "GET /api/analytics/activity",
    "GET /api/analytics/activity/project/{project_id}",

    # Admin
    "GET /api/admin/overview",
    "GET /api/admin/users",
    "GET /api/admin/ai-usage",
    "GET /api/admin/system",

    # Recommendations
    "GET /api/recommendations",
    "POST /api/recommendations/generate/{project_id}",
}


# Collect routes directly from the APIRouter objects that were loaded.
registered_api_routes = set()

for router_name, router_module in loaded_api_modules.items():

    router = getattr(router_module, "router", None)

    if router is None:
        print(f"⚠ {router_name}: no router object")
        continue

    for route in router.routes:

        path = getattr(route, "path", None)
        methods = getattr(route, "methods", None)

        if not path or not methods:
            continue

        full_path = "/api" + path

        for method in methods:
            registered_api_routes.add(
                f"{method.upper()} {full_path}"
            )


# ------------------------------------------------------------------
# Compare expected vs actual
# ------------------------------------------------------------------

missing_routes = expected_api_routes - registered_api_routes
unexpected_routes = registered_api_routes - expected_api_routes


print()
print(f"API endpoints discovered: {len(registered_api_routes)}")
print()


if missing_routes:
    print("MISSING ROUTES:")
    for route in sorted(missing_routes):
        print(f"  {route}")

    raise RuntimeError(
        "Required API routes are missing:\n"
        + "\n".join(
            f"  {route}"
            for route in sorted(missing_routes)
        )
    )


if unexpected_routes:
    print("Additional registered routes:")
    for route in sorted(unexpected_routes):
        print(f"  {route}")

print("✓ Authentication routes")
print("✓ Spaces routes")
print("✓ Projects routes")
print("✓ Materials routes")
print("✓ Tutor routes")
print("✓ Quiz routes")
print("✓ Mastery routes")
print("✓ Growth routes")
print("✓ Analytics routes")
print("✓ Admin routes")
print("✓ Recommendations routes")

print()
print(f"✓ {len(registered_api_routes)} API endpoints registered")
print(f"✓ FastAPI app contains {len(app.routes)} top-level routes")
print("✓ ROUTE VALIDATION PASSED")

AI STUDY COMPANION — BACKEND LAUNCHER
Project: /content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack

LOADING BACKEND MODULES

Loading core...
✓ core loaded

Loading utils...
✓ utils loaded

Loading models...
Model: Activity
Scope: User + optional Project
Purpose: Store learning and system events
Example events:
- PROJECT_CREATED
- MATERIAL_UPLOADED
- TUTOR_MESSAGE_SENT
- QUIZ_COMPLETED
- MASTERY_UPDATED
- RECOMMENDATION_CREATED
Model: AIUsage
Purpose: Track AI calls and observability metrics
Tracks: model, tokens, latency, cost, success/failure
Tutor metrics: retrieval count, citations, groundedness
Model: Assessment
Nested model: AssessmentQuestion
Nested model: AssessmentAnswer
Supported question types: MCQ, open-ended
Supported assessment types: adaptive_quiz, practice, assessment
Purpose: Store assessment questions, student answers, and evaluation
Model: Conversation
Nested model: Message
Nested model: Citation
Conversation belongs to: Project + User
Purp

In [5]:
# ================================================================
# START FASTAPI SERVER — COLAB/JUPYTER
# ================================================================

print()
print("=" * 70)
print("STARTING FASTAPI SERVER")
print("=" * 70)

import socket
import threading
import time
import uvicorn


HOST = "0.0.0.0"
PORT = 8000


# ------------------------------------------------
# Check port
# ------------------------------------------------

def port_is_open(host, port):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(1)

    try:
        return sock.connect_ex((host, port)) == 0
    finally:
        sock.close()


# ------------------------------------------------
# Start Uvicorn in a background thread.
#
# This is important for Google Colab:
#
#   Main thread  -> Jupyter/Colab event loop
#   Background   -> Uvicorn/FastAPI
#
# Therefore the notebook remains usable while
# the FastAPI server continues running.
# ------------------------------------------------

# ------------------------------------------------
# A previous run of this launcher may still be
# serving an OLD app object (old routes, old CORS
# middleware). Stop it so the app built above is
# always the one being served.
# ------------------------------------------------

previous_server = globals().get("server")
previous_thread = globals().get("server_thread")

if previous_server is not None:

    print("Stopping previous launcher server...")

    previous_server.should_exit = True
    previous_server.force_exit = True

    if previous_thread is not None:
        previous_thread.join(timeout=10)

    for attempt in range(40):
        if not port_is_open("127.0.0.1", PORT):
            break
        time.sleep(0.25)

    server = None
    server_thread = None


if port_is_open("127.0.0.1", PORT):

    raise RuntimeError(
        f"Port {PORT} is held by another process. "
        "Stop it (or restart the runtime) before launching, "
        "otherwise the browser will keep talking to a stale app."
    )

else:

    uvicorn_config = uvicorn.Config(
        app,
        host=HOST,
        port=PORT,
        log_level="info",
    )

    server = uvicorn.Server(uvicorn_config)

    def run_uvicorn():
        server.run()

    server_thread = threading.Thread(
        target=run_uvicorn,
        daemon=True,
        name="fastapi-uvicorn",
    )

    server_thread.start()

    # Give Uvicorn time to bind to port 8000.
    for attempt in range(20):

        if port_is_open("127.0.0.1", PORT):
            break

        time.sleep(0.25)

    # ------------------------------------------------
    # Verify the server actually started.
    # ------------------------------------------------

    if not port_is_open("127.0.0.1", PORT):
        raise RuntimeError(
            "FastAPI failed to start on port 8000."
        )

    print("✓ Uvicorn started successfully")
    print(f"✓ Server thread alive: {server_thread.is_alive()}")
    print(f"✓ API:  http://127.0.0.1:{PORT}")
    print(f"✓ Docs: http://127.0.0.1:{PORT}/docs")

print()
print("=" * 70)
print("FASTAPI SERVER READY")
print("=" * 70)


STARTING FASTAPI SERVER


INFO:     Started server process [35794]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✓ Uvicorn started successfully
✓ Server thread alive: True
✓ API:  http://127.0.0.1:8000
✓ Docs: http://127.0.0.1:8000/docs

FASTAPI SERVER READY


In [6]:
# ================================================================
# AI STUDY COMPANION — FULL BACKEND INTEGRATION TEST
# ================================================================
#
# Run this in a SEPARATE Colab cell while run_backend.ipynb
# is running on port 8000.
#
# Tests:
#   1. Health
#   2. Authentication + JWT
#   3. Spaces CRUD
#   4. Projects CRUD
#   5. Materials
#   6. Tutor
#   7. Quiz generation/start/answer/complete
#   8. Mastery
#   9. Growth
#  10. Analytics
#  11. Admin authorization
#
# This test creates temporary test data in MongoDB.
# ================================================================

import requests
import uuid
import json
import time
import traceback
from datetime import datetime, timezone

BASE_URL = "http://127.0.0.1:8000"

# ----------------------------------------------------------------
# TEST CONFIGURATION
# ----------------------------------------------------------------

TEST_ID = uuid.uuid4().hex[:12]

TEST_EMAIL = f"backend_test_{TEST_ID}@example.com"
TEST_PASSWORD = "TestPassword123!"

TEST_USER = {
    "email": TEST_EMAIL,
    "password": TEST_PASSWORD,
    "name": f"Backend Test User {TEST_ID}",
}

SPACE_NAME = f"Integration Test Space {TEST_ID}"
PROJECT_NAME = f"Integration Test Project {TEST_ID}"


# ----------------------------------------------------------------
# TEST STATE
# ----------------------------------------------------------------

results = []

token = None
headers = {}

space_id = None
project_id = None
material_id = None
assessment_id = None
conversation_id = None


# ----------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------

def record(name, passed, detail=""):
    results.append({
        "name": name,
        "passed": passed,
        "detail": detail,
    })

    symbol = "✓" if passed else "✗"

    print(f"{symbol} {name}")

    if detail:
        print(f"    {detail}")


def response_detail(response):
    try:
        body = response.json()
        return json.dumps(body, indent=2, default=str)[:1500]
    except Exception:
        return response.text[:1500]

def request(
    method,
    path,
    *,
    expected=None,
    json_body=None,
    data=None,
    files=None,
    params=None,
    auth=True,
    timeout=120,
):
    url = BASE_URL + path

    request_headers = {}

    if auth and token:
        request_headers["Authorization"] = f"Bearer {token}"

    try:
        response = requests.request(
            method,
            url,
            headers=request_headers,
            params=params,
            json=json_body,
            data=data,
            files=files,
            timeout=timeout,
        )

        if expected is not None:
            if isinstance(expected, int):
                expected_codes = {expected}
            else:
                expected_codes = set(expected)

            if response.status_code not in expected_codes:
                raise AssertionError(
                    f"Expected HTTP {sorted(expected_codes)}, "
                    f"got {response.status_code}\n"
                    f"{response_detail(response)}"
                )

        return response

    except Exception:
        raise

def run_test(name, fn):
    try:
        value = fn()
        record(name, True)
        return value

    except Exception as exc:
        record(
            name,
            False,
            f"{type(exc).__name__}: {exc}",
        )
        return None


def extract_id(payload, *keys):
    if not isinstance(payload, dict):
        return None

    for key in keys:
        value = payload.get(key)

        if value:
            return value

    return None


# ================================================================
# START
# ================================================================

print("=" * 70)
print("AI STUDY COMPANION — FULL BACKEND INTEGRATION TEST")
print("=" * 70)
print(f"Base URL: {BASE_URL}")
print(f"Test ID:  {TEST_ID}")
print("=" * 70)


# ================================================================
# 1. HEALTH
# ================================================================

print("\n" + "=" * 70)
print("1. HEALTH CHECK")
print("=" * 70)


def test_health():
    response = request(
        "GET",
        "/health",
        expected=200,
        auth=False,
    )

    body = response.json()

    assert body.get("status") == "ok", body

    return body


health = run_test("GET /health", test_health)


# ================================================================
# 2. AUTHENTICATION + JWT
# ================================================================

print("\n" + "=" * 70)
print("2. AUTHENTICATION + JWT")
print("=" * 70)


def test_register():
    response = request(
        "POST",
        "/api/auth/register",
        expected={200, 201},
        json_body=TEST_USER,
        auth=False,
    )

    body = response.json()

    return body


register_body = run_test(
    "POST /api/auth/register",
    test_register,
)


def test_login():
    response = request(
        "POST",
        "/api/auth/login",
        expected=200,
        json_body={
            "email": TEST_EMAIL,
            "password": TEST_PASSWORD,
        },
        auth=False,
    )

    body = response.json()

    global token, headers

    token = (
        body.get("access_token")
        or body.get("token")
    )

    assert token, (
        "Login succeeded but no access token was returned:\n"
        + response_detail(response)
    )

    headers = {
        "Authorization": f"Bearer {token}"
    }

    return body


login_body = run_test(
    "POST /api/auth/login",
    test_login,
)


def test_me():
    assert token, "No JWT available from login"

    response = request(
        "GET",
        "/api/auth/me",
        expected=200,
    )

    body = response.json()

    assert isinstance(body, dict)

    return body


me_body = run_test(
    "GET /api/auth/me",
    test_me,
)


def test_invalid_jwt():
    response = requests.get(
        BASE_URL + "/api/auth/me",
        headers={
            "Authorization": "Bearer invalid.jwt.token"
        },
        timeout=30,
    )

    assert response.status_code == 401, (
        f"Expected 401, got {response.status_code}\n"
        f"{response_detail(response)}"
    )


run_test(
    "Invalid JWT rejected with 401",
    test_invalid_jwt,
)


# ================================================================
# 3. SPACES
# ================================================================

print("\n" + "=" * 70)
print("3. SPACES")
print("=" * 70)


def test_create_space():
    response = request(
        "POST",
        "/api/spaces",
        expected={200, 201},
        json_body={
            "name": SPACE_NAME,
            "description": "Backend integration test space",
        },
    )

    body = response.json()

    global space_id

    space_id = extract_id(
        body,
        "id",
        "_id",
        "space_id",
    )

    assert space_id, (
        "Space was created but no ID was returned:\n"
        + response_detail(response)
    )

    return body


space_body = run_test(
    "POST /api/spaces",
    test_create_space,
)


def test_list_spaces():
    response = request(
        "GET",
        "/api/spaces",
        expected=200,
    )

    body = response.json()

    assert isinstance(body, (list, dict))

    return body


run_test(
    "GET /api/spaces",
    test_list_spaces,
)


def test_get_space():
    assert space_id, "Space creation did not return an ID"

    response = request(
        "GET",
        f"/api/spaces/{space_id}",
        expected=200,
    )

    return response.json()


run_test(
    "GET /api/spaces/{space_id}",
    test_get_space,
)


def test_update_space():
    assert space_id, "Space creation did not return an ID"

    response = request(
        "PUT",
        f"/api/spaces/{space_id}",
        expected=200,
        json_body={
            "name": SPACE_NAME + " Updated",
            "description": "Updated integration test space",
        },
    )

    return response.json()


run_test(
    "PUT /api/spaces/{space_id}",
    test_update_space,
)


# ================================================================
# 4. PROJECTS
# ================================================================

print("\n" + "=" * 70)
print("4. PROJECTS")
print("=" * 70)


def test_create_project():
    assert space_id, "Cannot create project without space"

    response = request(
        "POST",
        "/api/projects",
        expected={200, 201},
        json_body={
            "name": PROJECT_NAME,
            "description": "Backend integration test project",
            "space_id": str(space_id),
        },
    )

    body = response.json()

    global project_id

    project_id = extract_id(
        body,
        "id",
        "_id",
        "project_id",
    )

    assert project_id, (
        "Project was created but no ID was returned:\n"
        + response_detail(response)
    )

    return body


project_body = run_test(
    "POST /api/projects",
    test_create_project,
)


def test_list_projects():
    response = request(
        "GET",
        "/api/projects",
        expected=200,
    )

    return response.json()


run_test(
    "GET /api/projects",
    test_list_projects,
)


def test_get_project():
    assert project_id, "Project creation did not return an ID"

    response = request(
        "GET",
        f"/api/projects/{project_id}",
        expected=200,
    )

    return response.json()


run_test(
    "GET /api/projects/{project_id}",
    test_get_project,
)


def test_update_project():
    assert project_id, "Project creation did not return an ID"

    response = request(
        "PUT",
        f"/api/projects/{project_id}",
        expected=200,
        json_body={
            "name": PROJECT_NAME + " Updated",
            "description": "Updated backend integration project",
        },
    )

    return response.json()


run_test(
    "PUT /api/projects/{project_id}",
    test_update_project,
)


# ================================================================
# 5. MATERIAL UPLOAD
# ================================================================

print("\n" + "=" * 70)
print("5. MATERIALS")
print("=" * 70)

def test_material_upload():
    assert project_id, "Cannot upload material without project"

    # Create a small, parser-compatible PDF containing actual text.
    # This is intentionally generated with pypdf so the document
    # processing pipeline can extract non-empty text.
    from io import BytesIO
    from pypdf import PdfWriter
    from reportlab.pdfgen import canvas

    pdf_buffer = BytesIO()

    pdf_canvas = canvas.Canvas(pdf_buffer)
    pdf_canvas.setFont("Helvetica", 12)
    pdf_canvas.drawString(
        72,
        720,
        "AI Study Companion integration test document."
    )
    pdf_canvas.drawString(
        72,
        700,
        "This document contains project knowledge for testing."
    )
    pdf_canvas.drawString(
        72,
        680,
        "The system should extract this text, create chunks, "
        "and generate embeddings."
    )
    pdf_canvas.save()

    pdf_bytes = pdf_buffer.getvalue()

    files = {
        "file": (
            "integration_test.pdf",
            pdf_bytes,
            "application/pdf",
        )
    }

    response = request(
        "POST",
        "/api/materials/upload",
        expected={200, 201, 202},
        params={
            "project_id": str(project_id),
        },
        files=files,
        timeout=180,
    )

    body = response.json()

    global material_id

    material_id = extract_id(
        body,
        "id",
        "_id",
        "material_id",
    )

    assert material_id, (
        "Material upload succeeded but no material ID "
        "was returned:\n"
        + response_detail(response)
    )

    return body

material_body = run_test(
    "POST /api/materials/upload",
    test_material_upload,
)

# ---------------------------------------------------------------
# Wait for document processing
# ---------------------------------------------------------------

def wait_for_material_processing(
    timeout_seconds=180,
    poll_interval=3,
):
    assert material_id, "Material ID unavailable"

    deadline = time.time() + timeout_seconds

    while time.time() < deadline:

        response = request(
            "GET",
            f"/api/materials/{material_id}",
            expected=200,
            timeout=30,
        )

        body = response.json()

        status_value = (
            body.get("processing_status")
            or body.get("status")
        )

        print(
            f"Material processing status: {status_value}"
        )

        if status_value == "completed":
            return body

        if status_value == "failed":
            raise AssertionError(
                "Material processing failed:\n"
                + json.dumps(
                    body,
                    indent=2,
                    default=str,
                )
            )

        time.sleep(poll_interval)

    raise AssertionError(
        f"Material processing did not complete "
        f"within {timeout_seconds} seconds."
    )


material_processing_body = run_test(
    "WAIT /api/materials/{material_id} processing",
    wait_for_material_processing,
)

def test_project_materials():
    assert project_id, "Project ID unavailable"

    response = request(
        "GET",
        f"/api/materials/project/{project_id}",
        expected=200,
    )

    return response.json()


materials_body = run_test(
    "GET /api/materials/project/{project_id}",
    test_project_materials,
)


# ================================================================
# 6. TUTOR
# ================================================================

print("\n" + "=" * 70)
print("6. AI TUTOR")
print("=" * 70)


def test_tutor():
    assert project_id, "Cannot test Tutor without project"

    response = request(
        "POST",
        "/api/tutor/ask",
        expected=200,
        json_body={
            "project_id": str(project_id),
            "message": (
                "Explain the main topic of this project. "
                "If there is insufficient project material, "
                "say that clearly."
            ),
        },
        timeout=180,
    )

    body = response.json()

    assert isinstance(body, dict)

    return body


tutor_body = run_test(
    "POST /api/tutor/ask",
    test_tutor,
)


def test_project_conversations():
    assert project_id, "Project ID unavailable"

    response = request(
        "GET",
        f"/api/tutor/conversations/{project_id}",
        expected=200,
    )

    body = response.json()

    global conversation_id

    if isinstance(body, list) and body:
        conversation_id = extract_id(
            body[0],
            "id",
            "_id",
            "conversation_id",
        )

    elif isinstance(body, dict):
        items = (
            body.get("conversations")
            or body.get("items")
            or []
        )

        if items:
            conversation_id = extract_id(
                items[0],
                "id",
                "_id",
                "conversation_id",
            )

    return body


run_test(
    "GET /api/tutor/conversations/{project_id}",
    test_project_conversations,
)


# ================================================================
# 7. QUIZ
# ================================================================

print("\n" + "=" * 70)
print("7. ADAPTIVE QUIZ")
print("=" * 70)

def test_generate_quiz():
    assert project_id, "Cannot generate quiz without project"

    response = request(
        "POST",
        "/api/quiz/generate",
        expected=200,
        json_body={
            "project_id": str(project_id),
            "question_count": 5,
            "difficulty": "medium",
        },
        timeout=180,
    )

    body = response.json()

    global assessment_id

    assessment_id = extract_id(
        body,
        "id",
        "_id",
        "assessment_id",
    )

    assert assessment_id, (
        "Quiz generation succeeded but no assessment ID "
        "was returned:\n"
        + response_detail(response)
    )

    questions = body.get("questions", [])

    assert isinstance(questions, list), (
        "Quiz response contains invalid questions field."
    )

    assert len(questions) > 0, (
        "Quiz generation returned an assessment with "
        "zero questions:\n"
        + json.dumps(
            body,
            indent=2,
            default=str,
        )
    )

    print(
        f"✓ Quiz generated with {len(questions)} questions"
    )

    return body
quiz_body = run_test(
    "POST /api/quiz/generate",
    test_generate_quiz,
)


def test_get_quiz():
    assert assessment_id, "Assessment ID unavailable"

    response = request(
        "GET",
        f"/api/quiz/{assessment_id}",
        expected=200,
    )

    return response.json()


quiz_get_body = run_test(
    "GET /api/quiz/{assessment_id}",
    test_get_quiz,
)


def test_start_quiz():
    assert assessment_id, "Assessment ID unavailable"

    response = request(
        "POST",
        f"/api/quiz/{assessment_id}/start",
        expected=200,
        json_body={},
    )

    return response.json()


quiz_start_body = run_test(
    "POST /api/quiz/{assessment_id}/start",
    test_start_quiz,
)


# ---------------------------------------------------------------
# Answer first question if available
# ---------------------------------------------------------------

def get_first_question_id():
    body = quiz_start_body or quiz_body or quiz_get_body

    if not isinstance(body, dict):
        return None

    questions = body.get("questions") or []

    if not questions:
        return None

    first = questions[0]

    if not isinstance(first, dict):
        return None

    return extract_id(
        first,
        "id",
        "question_id",
    )


def test_answer_quiz():
    question_id = get_first_question_id()

    if not question_id:
        raise AssertionError(
            "No question ID available in generated quiz"
        )

    response = request(
        "POST",
        f"/api/quiz/{assessment_id}/answer",
        expected=200,
        json_body={
            "question_id": question_id,
            "answer": "A",
        },
    )

    return response.json()


run_test(
    "POST /api/quiz/{assessment_id}/answer",
    test_answer_quiz,
)


def test_complete_quiz():
    assert assessment_id, "Assessment ID unavailable"

    response = request(
        "POST",
        f"/api/quiz/{assessment_id}/complete",
        expected=200,
        json_body={},
    )

    return response.json()


quiz_complete_body = run_test(
    "POST /api/quiz/{assessment_id}/complete",
    test_complete_quiz,
)


# ================================================================
# 8. MASTERY
# ================================================================

print("\n" + "=" * 70)
print("8. MASTERY")
print("=" * 70)


def test_mastery():
    assert project_id, "Project ID unavailable"

    response = request(
        "GET",
        f"/api/mastery/project/{project_id}",
        expected=200,
    )

    return response.json()


mastery_body = run_test(
    "GET /api/mastery/project/{project_id}",
    test_mastery,
)


# ================================================================
# 9. GROWTH
# ================================================================

print("\n" + "=" * 70)
print("9. GROWTH")
print("=" * 70)


def test_growth():
    assert project_id, "Project ID unavailable"

    response = request(
        "GET",
        f"/api/growth/project/{project_id}",
        expected=200,
    )

    return response.json()


growth_body = run_test(
    "GET /api/growth/project/{project_id}",
    test_growth,
)


# ================================================================
# 10. ANALYTICS
# ================================================================

print("\n" + "=" * 70)
print("10. ANALYTICS")
print("=" * 70)


def test_project_analytics():
    assert project_id, "Project ID unavailable"

    response = request(
        "GET",
        f"/api/analytics/project/{project_id}",
        expected=200,
    )

    return response.json()


analytics_project_body = run_test(
    "GET /api/analytics/project/{project_id}",
    test_project_analytics,
)


def test_activity():
    response = request(
        "GET",
        "/api/analytics/activity",
        expected=200,
    )

    return response.json()


run_test(
    "GET /api/analytics/activity",
    test_activity,
)


def test_project_activity():
    assert project_id, "Project ID unavailable"

    response = request(
        "GET",
        f"/api/analytics/activity/project/{project_id}",
        expected=200,
    )

    return response.json()


run_test(
    "GET /api/analytics/activity/project/{project_id}",
    test_project_activity,
)


# ================================================================
# 11. ADMIN AUTHORIZATION
# ================================================================

print("\n" + "=" * 70)
print("11. ADMIN AUTHORIZATION")
print("=" * 70)


def test_admin_overview_for_student():
    response = request(
        "GET",
        "/api/admin/overview",
        expected=403,
    )

    return response.json()


run_test(
    "Student blocked from GET /api/admin/overview",
    test_admin_overview_for_student,
)


def test_admin_users_for_student():
    response = request(
        "GET",
        "/api/admin/users",
        expected=403,
    )

    return response.json()


run_test(
    "Student blocked from GET /api/admin/users",
    test_admin_users_for_student,
)


def test_admin_ai_usage_for_student():
    response = request(
        "GET",
        "/api/admin/ai-usage",
        expected=403,
    )

    return response.json()


run_test(
    "Student blocked from GET /api/admin/ai-usage",
    test_admin_ai_usage_for_student,
)


def test_admin_system_for_student():
    response = request(
        "GET",
        "/api/admin/system",
        expected=403,
    )

    return response.json()


run_test(
    "Student blocked from GET /api/admin/system",
    test_admin_system_for_student,
)


# ================================================================
# FINAL SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("FINAL INTEGRATION TEST SUMMARY")
print("=" * 70)

passed = sum(1 for result in results if result["passed"])
failed = sum(1 for result in results if not result["passed"])
total = len(results)

print()
print(f"Total tests : {total}")
print(f"Passed      : {passed}")
print(f"Failed      : {failed}")
print()

if failed:
    print("FAILED TESTS")
    print("-" * 70)

    for result in results:
        if not result["passed"]:
            print()
            print(f"✗ {result['name']}")
            print(f"  {result['detail']}")

    print()
    print("=" * 70)
    print("INTEGRATION TEST FAILED")
    print("=" * 70)

else:
    print("=" * 70)
    print("✓ ALL BACKEND INTEGRATION TESTS PASSED")
    print("=" * 70)

print()
print("Test data:")
print(f"  Email      : {TEST_EMAIL}")
print(f"  Space ID   : {space_id}")
print(f"  Project ID : {project_id}")
print(f"  Material   : {material_id}")
print(f"  Assessment : {assessment_id}")
print(f"  Conversation: {conversation_id}")
print("=" * 70)

AI STUDY COMPANION — FULL BACKEND INTEGRATION TEST
Base URL: http://127.0.0.1:8000
Test ID:  2592fc1e5298

1. HEALTH CHECK
INFO:     127.0.0.1:33172 - "GET /health HTTP/1.1" 200 OK
✓ GET /health

2. AUTHENTICATION + JWT
INFO:     127.0.0.1:33188 - "POST /api/auth/register HTTP/1.1" 201 Created
✓ POST /api/auth/register
INFO:     127.0.0.1:33204 - "POST /api/auth/login HTTP/1.1" 200 OK
✓ POST /api/auth/login
INFO:     127.0.0.1:33214 - "GET /api/auth/me HTTP/1.1" 200 OK
✓ GET /api/auth/me
INFO:     127.0.0.1:33220 - "GET /api/auth/me HTTP/1.1" 401 Unauthorized
✓ Invalid JWT rejected with 401

3. SPACES
INFO:     127.0.0.1:33228 - "POST /api/spaces HTTP/1.1" 200 OK
✓ POST /api/spaces
INFO:     127.0.0.1:44150 - "GET /api/spaces HTTP/1.1" 200 OK
✓ GET /api/spaces
INFO:     127.0.0.1:44166 - "GET /api/spaces/d1c78fc9-7378-4ee9-b0ca-786d7f6a10a9 HTTP/1.1" 200 OK
✓ GET /api/spaces/{space_id}
INFO:     127.0.0.1:44174 - "PUT /api/spaces/d1c78fc9-7378-4ee9-b0ca-786d7f6a10a9 HTTP/1.1" 200 OK
✓ 

INFO:ai_study_companion:material.processing.started | material_id=af96fac2-ae48-49a0-831e-2abba66658d1 file_name=integration_test.pdf


2026-09-17 05:18:57,236 | INFO | ai_study_companion | material.extraction.completed | material_id=af96fac2-ae48-49a0-831e-2abba66658d1 page_count=1 elapsed_seconds=0.01


INFO:ai_study_companion:material.extraction.completed | material_id=af96fac2-ae48-49a0-831e-2abba66658d1 page_count=1 elapsed_seconds=0.01


2026-09-17 05:18:57,238 | INFO | ai_study_companion | material.chunking.completed | material_id=af96fac2-ae48-49a0-831e-2abba66658d1 chunk_count=1


...

2026-09-17 05:19:00,881 | INFO | ai_study_companion | material.processing.completed | material_id=af96fac2-ae48-49a0-831e-2abba66658d1 page_count=1 chunk_count=1 elapsed_seconds=3.66


INFO:ai_study_companion:material.processing.completed | material_id=af96fac2-ae48-49a0-831e-2abba66658d1 page_count=1 chunk_count=1 elapsed_seconds=3.66


INFO:     127.0.0.1:59318 - "GET /api/materials/af96fac2-ae48-49a0-831e-2abba66658d1 HTTP/1.1" 200 OK
Material processing status: completed
✓ WAIT /api/materials/{material_id} processing
INFO:     127.0.0.1:59332 - "GET /api/materials/project/a13b843d-17ac-48fd-98b5-7cf3d408280d HTTP/1.1" 200 OK
✓ GET /api/materials/project/{project_id}

6. AI TUTOR


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:     127.0.0.1:59338 - "POST /api/tutor/ask HTTP/1.1" 200 OK
✓ POST /api/tutor/ask
INFO:     127.0.0.1:59352 - "GET /api/tutor/conversations/a13b843d-17ac-48fd-98b5-7cf3d408280d HTTP/1.1" 404 Not Found
✗ GET /api/tutor/conversations/{project_id}
    AssertionError: Expected HTTP [200], got 404
{
  "detail": "Conversation not found."
}

7. ADAPTIVE QUIZ
[QuizGenerator] evidence_chunks=1 approx_prompt_tokens=271 question_count=5
[AIService] operation=quiz_generation prompt_chars=2186 approx_input_tokens=547
INFO:     127.0.0.1:59368 - "POST /api/quiz/generate HTTP/1.1" 200 OK
✓ Quiz generated with 5 questions
✓ POST /api/quiz/generate
INFO:     127.0.0.1:53492 - "GET /api/quiz/5441de08-16d9-417e-81d8-46e9dce38b4a HTTP/1.1" 200 OK
✓ GET /api/quiz/{assessment_id}
INFO:     127.0.0.1:53494 - "POST /api/quiz/5441de08-16d9-417e-81d8-46e9dce38b4a/start HTTP/1.1" 200 OK
✓ POST /api/quiz/{assessment_id}/start
INFO:     127.0.0.1:53500 - "POST /api/quiz/5441de08-16d9-417e-81d8-46e9dce38b4a/a

In [7]:
!pip install -q pyngrok

In [8]:
from pyngrok import ngrok

ngrok.kill()

print("Existing ngrok tunnels killed.")

Existing ngrok tunnels killed.


In [9]:
import requests

r = requests.get("http://127.0.0.1:8000/health", timeout=10)

print("FastAPI:", r.status_code)
print(r.text)

INFO:     127.0.0.1:38046 - "GET /health HTTP/1.1" 200 OK
FastAPI: 200
{"status":"ok","service":"ai-study-companion"}


In [10]:
from google.colab import userdata
from pyngrok import ngrok

token = userdata.get("NGROK_AUTH_TOKEN")
ngrok.set_auth_token(token)
print("NGROK authentication configured:", bool(token))
tunnel = ngrok.connect(
    addr="8000",
    proto="http",
)

print("PUBLIC URL:")
print(tunnel.public_url)

print("\nACTIVE TUNNELS:")
for t in ngrok.get_tunnels():
    print(t.public_url, "->", t.config.get("addr"))

NGROK authentication configured: True
PUBLIC URL:
https://underuse-makeover-reminder.ngrok-free.dev

ACTIVE TUNNELS:
https://underuse-makeover-reminder.ngrok-free.dev -> http://localhost:8000


In [11]:
from pathlib import Path
import os

print("=" * 70)
print("BACKEND PATH DIAGNOSTIC")
print("=" * 70)

print("Current working directory:")
print(os.getcwd())

print("\nPROJECT_ROOT:")
print(PROJECT_ROOT)

print("\nPROJECT_ROOT exists:", PROJECT_ROOT.exists())
print("PROJECT_ROOT is directory:", PROJECT_ROOT.is_dir())

print("\nBACKEND_DIR:")
print(BACKEND_DIR)

print("BACKEND_DIR exists:", BACKEND_DIR.exists())
print("BACKEND_DIR is directory:", BACKEND_DIR.is_dir())

print("\nExpected backend files:")

expected_files = [
    "backend/app/core/config.ipynb",
    "backend/app/core/security.ipynb",
    "backend/app/core/database.ipynb",

    "backend/app/ai/prompts.ipynb",
    "backend/app/ai/tutor.ipynb",
    "backend/app/ai/quiz_generator.ipynb",
    "backend/app/ai/evaluator.ipynb",
    "backend/app/ai/recommender.ipynb",

    "backend/app/services/ai_service.ipynb",
    "backend/app/services/document_service.ipynb",
    "backend/app/services/retrieval_service.ipynb",
    "backend/app/services/tutor_service.ipynb",
    "backend/app/services/quiz_service.ipynb",
    "backend/app/services/recommendation_service.ipynb",

    "backend/app/workers/document_worker.ipynb",
    "backend/app/workers/learning_worker.ipynb",
    "backend/app/workers/quiz_worker.ipynb",

    "backend/app/main.ipynb",
]

for relative in expected_files:
    path = PROJECT_ROOT / relative

    status = "✓ EXISTS" if path.exists() else "✗ MISSING"

    print(f"{status:12} {path}")

print("\n" + "=" * 70)
print("BACKEND DIRECTORY TREE")
print("=" * 70)

if BACKEND_DIR.exists():
    for path in sorted(BACKEND_DIR.rglob("*.ipynb")):
        print(path)

print("=" * 70)

BACKEND PATH DIAGNOSTIC
Current working directory:
/content

PROJECT_ROOT:
/content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack

PROJECT_ROOT exists: True
PROJECT_ROOT is directory: True

BACKEND_DIR:
/content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/backend
BACKEND_DIR exists: True
BACKEND_DIR is directory: True

Expected backend files:
✓ EXISTS     /content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/backend/app/core/config.ipynb
✓ EXISTS     /content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/backend/app/core/security.ipynb
✓ EXISTS     /content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/backend/app/core/database.ipynb
✓ EXISTS     /content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/backend/app/ai/prompts.ipynb
✓ EXISTS     /content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/backend/app/ai/

In [12]:
activities = database.collection("activities")

print("ACTIVITIES:", activities.count_documents({}))

for activity in activities.find({}, {"_id": 0}).sort("created_at", -1).limit(10):
    print(activity)

ACTIVITIES: 0


In [13]:
!curl -i http://127.0.0.1:8000/health

INFO:     127.0.0.1:38048 - "GET /health HTTP/1.1" 200 OK
HTTP/1.1 200 OK
date: Thu, 17 Sep 2026 05:19:34 GMT
server: uvicorn
content-length: 46
content-type: application/json

{"status":"ok","service":"ai-study-companion"}

In [14]:
import requests

public_url = tunnel.public_url

r = requests.get(
    f"{public_url}/health",
    headers={"ngrok-skip-browser-warning": "true"},
    timeout=20,
)

print("STATUS:", r.status_code)
print("CONTENT-TYPE:", r.headers.get("content-type"))
print("BODY:", r.text)

INFO:     34.125.230.109:0 - "GET /health HTTP/1.1" 200 OK
STATUS: 200
CONTENT-TYPE: application/json
BODY: {"status":"ok","service":"ai-study-companion"}


In [15]:
from pyngrok import ngrok
import requests

print("Tunnels before:")
for t in ngrok.get_tunnels():
    print(t.public_url, "->", t.config.get("addr"))

public_url = ngrok.get_tunnels()[0].public_url

r = requests.get(
    f"{public_url}/health",
    headers={"ngrok-skip-browser-warning": "true"},
    timeout=20,
)

print("\nURL:", public_url)
print("STATUS:", r.status_code)
print("CONTENT-TYPE:", r.headers.get("content-type"))
print("BODY:", r.text)

Tunnels before:
https://underuse-makeover-reminder.ngrok-free.dev -> http://localhost:8000
INFO:     34.125.230.109:0 - "GET /health HTTP/1.1" 200 OK

URL: https://underuse-makeover-reminder.ngrok-free.dev
STATUS: 200
CONTENT-TYPE: application/json
BODY: {"status":"ok","service":"ai-study-companion"}


In [16]:
import requests

r = requests.get(
    "http://127.0.0.1:8000/api/auth/me",
    headers={
        "Origin": "http://localhost:3000",
    },
)

print("STATUS:", r.status_code)
print("ACCESS-CONTROL-ALLOW-ORIGIN:",
      r.headers.get("access-control-allow-origin"))
print("BODY:", r.text)

INFO:     127.0.0.1:38078 - "GET /api/auth/me HTTP/1.1" 401 Unauthorized
STATUS: 401
ACCESS-CONTROL-ALLOW-ORIGIN: http://localhost:3000
BODY: {"detail":"Not authenticated"}


In [17]:
from app.main import app

print("CORS middleware check:")

for middleware in app.user_middleware:
    print(middleware)

print("Middleware count:", len(app.user_middleware))

for i, middleware in enumerate(app.user_middleware):
    print(i, middleware.cls)

CORS middleware check:
Middleware(CORSMiddleware, allow_origins=['http://localhost:3000', 'http://127.0.0.1:3000'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'], expose_headers=['*'])
Middleware count: 1
0 <class 'starlette.middleware.cors.CORSMiddleware'>


In [18]:
import os

print("CORS_ORIGINS =", repr(os.environ.get("CORS_ORIGINS")))
from app.main import cors_origins

print("ACTIVE CORS ORIGINS:")
for origin in cors_origins():
    print(repr(origin))

CORS_ORIGINS = None
ACTIVE CORS ORIGINS:
'http://localhost:3000'
'http://127.0.0.1:3000'


In [19]:
import requests

BASE_URL = "http://127.0.0.1:8000"

token = input("Paste your JWT token: ").strip()

headers = {
    "Authorization": f"Bearer {token}"
}

r = requests.get(
    f"{BASE_URL}/api/analytics/activity",
    headers=headers
)

print("STATUS:", r.status_code)
print("RESPONSE:")
print(r.text[:5000])
project_id = "f6b016f0-3ea3-44e7-bba7-d1a0892061f9"

r = requests.get(
    f"{BASE_URL}/api/analytics/activity/project/{project_id}",
    headers=headers
)

print("STATUS:", r.status_code)
print("RESPONSE:")
print(r.text[:5000])

Paste your JWT token: ytru
INFO:     127.0.0.1:39494 - "GET /api/analytics/activity HTTP/1.1" 401 Unauthorized
STATUS: 401
RESPONSE:
{"detail":"Invalid or expired token."}
INFO:     127.0.0.1:39498 - "GET /api/analytics/activity/project/f6b016f0-3ea3-44e7-bba7-d1a0892061f9 HTTP/1.1" 401 Unauthorized
STATUS: 401
RESPONSE:
{"detail":"Invalid or expired token."}


In [20]:
db = app.state.database

print("ACTIVITIES:", db.collection("activities").count_documents({}))
print("PROJECTS:", db.collection("projects").count_documents({}))
print("MATERIALS:", db.collection("materials").count_documents({}))
print("CONVERSATIONS:", db.collection("conversations").count_documents({}))
print("ASSESSMENTS:", db.collection("assessments").count_documents({}))
print("MASTERY:", db.collection("mastery").count_documents({}))
print("RECOMMENDATIONS:", db.collection("recommendations").count_documents({}))

print("\nSample activity documents:")
for doc in db.collection("activities").find({}).limit(10):
    print(doc)

print("\nAvailable service modules:")
for name in loaded_services:
    print(name)

ACTIVITIES: 0
PROJECTS: 47
MATERIALS: 28
CONVERSATIONS: 37
ASSESSMENTS: 46
MASTERY: 15
RECOMMENDATIONS: 88

Sample activity documents:

Available service modules:


NameError: name 'loaded_services' is not defined

In [21]:
print("=== ACTIVITY WRITER SEARCH ===")

import os

backend_root = "/content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/backend"

for root, dirs, files in os.walk(backend_root):
    dirs[:] = [d for d in dirs if d != "__pycache__"]

    for filename in files:
        if filename.endswith(".ipynb"):
            path = os.path.join(root, filename)

            try:
                with open(path, "r", encoding="utf-8") as f:
                    text = f.read()

                if (
                    "activities" in text
                    or "Activity(" in text
                    or "MATERIAL_UPLOADED" in text
                    or "TUTOR_MESSAGE_SENT" in text
                    or "QUIZ_COMPLETED" in text
                    or "MASTERY_UPDATED" in text
                    or "RECOMMENDATION_CREATED" in text
                ):
                    print("\nFILE:", path)

                    for term in [
                        "activities",
                        "Activity(",
                        "MATERIAL_UPLOADED",
                        "TUTOR_MESSAGE_SENT",
                        "QUIZ_COMPLETED",
                        "MASTERY_UPDATED",
                        "RECOMMENDATION_CREATED",
                    ]:
                        if term in text:
                            print("  contains:", term)

            except Exception as e:
                print("ERROR:", path, e)

=== ACTIVITY WRITER SEARCH ===

FILE: /content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/backend/app/api/growth.ipynb
  contains: activities

FILE: /content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/backend/app/api/analytics.ipynb
  contains: activities

FILE: /content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/backend/app/api/admin.ipynb
  contains: activities

FILE: /content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/backend/app/core/database.ipynb
  contains: activities

FILE: /content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/backend/app/services/analytics_service.ipynb
  contains: activities

FILE: /content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/backend/app/services/recommendation_service.ipynb
  contains: activities

FILE: /content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/backen

In [ ]:
import os
import json

backend_root = "/content/drive/MyDrive/ai-study-companion-fixed-v2/ai-study-companion-fullstack/backend/app"

targets = [
    "api/projects.ipynb",
    "api/materials.ipynb",
    "api/tutor.ipynb",
    "api/quiz.ipynb",
    "services/mastery_service.ipynb",
    "services/recommendation_service.ipynb",
]

terms = [
    "insert_one",
    "update_one",
    "find_one",
    "async def",
    "def ",
]

for relative_path in targets:
    path = os.path.join(backend_root, relative_path)

    print("\n" + "=" * 80)
    print(relative_path)
    print("=" * 80)

    if not os.path.exists(path):
        print("MISSING:", path)
        continue

    with open(path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        if any(term in source for term in terms):
            print(f"\n--- CELL {i} ---")
            print(source)